# OptimumDebt Bellman Stages


## 1. The Paper and Its Model

### Paper

**Aiyagari, S. Rao, and Ellen R. McGrattan (1998), "The Optimum Quantity of Debt," *Journal of Monetary Economics* 42: 447–469.**

The paper calculates the welfare-maximizing level of risk-free government debt in a Bewley–Aiyagari incomplete-markets economy where infinitely-lived households face uninsurable idiosyncratic labor-productivity shocks and borrowing constraints.

The paper presents two model variants. We analyze the **benchmark elastic-labor-supply version** (Section III of the paper), because it features multiple within-period controls and is therefore suitable for a stage decomposition.

### Agent's objective

Each household maximizes expected discounted lifetime utility over consumption $\tilde{c}_t$ and leisure $l_t$, with Cobb-Douglas aggregation:

$$\max_{\{\tilde{c}_t,\, l_t,\, \tilde{a}_{t+1}\}} \; E_0 \left[\sum_{t=0}^{\infty} \tilde{\beta}^{\,t} \; \frac{\left(\tilde{c}_t^{\;\eta}\, l_t^{\;1-\eta}\right)^{1-\mu}}{1-\mu} \;\Bigg|\; \tilde{a}_0,\, e_0 \right]$$

where $\tilde{\beta} \equiv \beta(1+g)^{\eta(1-\mu)-1}$ is the growth-adjusted discount factor, $\eta$ is the consumption weight, and $\mu$ governs risk aversion over the composite good.

### State variables (at the start of period $t$)

| Variable | Description |
|----------|-------------|
| $\tilde{a}_t$ | Normalized beginning-of-period assets (wealth carried from $t-1$) |
| $e_t$ | Idiosyncratic labor-productivity state (discrete Markov chain) |

### Control variables (chosen within period $t$)

| Variable | Description |
|----------|-------------|
| $\tilde{c}_t$ | Consumption |
| $l_t$ | Leisure (equivalently, labor supply $h_t = 1 - l_t$) |

Next-period assets $\tilde{a}_{t+1}$ are then pinned down by the budget constraint.

### Shocks

The only idiosyncratic shock is **labor productivity** $e_t$, which follows a discrete Markov chain with transition probabilities $\pi_{e,e'}$. It is realized at the **beginning of period $t$**, before the household chooses $\tilde{c}_t$ and $l_t$. Prices $(\bar{r}, \bar{w})$ are taken as given by the household (determined in general equilibrium but constant from the individual's perspective).

### Budget constraint

$$\tilde{c}_t + (1+g)\,\tilde{a}_{t+1} \;\leq\; (1+\bar{r})\,\tilde{a}_t + \bar{w}\,e_t\,(1 - l_t) + \chi$$

with $\tilde{c}_t \geq 0$, $\;\tilde{a}_{t+1} \geq 0$, $\;0 \leq l_t \leq 1$.

Here $\bar{r}$ is the net interest rate, $\bar{w}$ is the wage, $e_t(1-l_t)$ is effective labor income, and $\chi$ is a lump-sum transfer.

### Bellman equation

$$V(\tilde{a},\, e) \;=\; \max_{\tilde{c},\, l} \left\{ \frac{\left(\tilde{c}^{\;\eta}\, l^{\;1-\eta}\right)^{1-\mu}}{1-\mu} \;+\; \tilde{\beta} \sum_{e'} \pi_{e,e'}\; V\!\left(\tilde{a}',\, e'\right) \right\}$$

subject to

$$\tilde{a}' = \frac{(1+\bar{r})\,\tilde{a} + \bar{w}\,e\,(1-l) + \chi - \tilde{c}}{1+g}$$

$$\tilde{c} \geq 0, \quad \tilde{a}' \geq 0, \quad 0 \leq l \leq 1$$

This is the **monolithic** formulation: the household simultaneously chooses both consumption $\tilde{c}$ and leisure $l$ to maximize the right-hand side. Sections 2–3 below will decompose this joint optimization into sequential stages.

### Notation and parameters

For reference, all symbols used throughout this notebook:

| Symbol | Meaning |
|--------|---------|
| $\tilde{a}_t$ | Normalized beginning-of-period assets |
| $e_t$ | Idiosyncratic labor-productivity state (Markov) |
| $\tilde{c}_t$ | Consumption |
| $l_t$ | Leisure ($h_t = 1 - l_t$ is labor supply) |
| $m_t$ | Total resources available for consumption and saving |
| $\tilde{a}'$ | End-of-period (next-period) assets |
| $\bar{r}$ | Net interest rate (taken as given by the household) |
| $\bar{w}$ | Wage rate per efficiency unit of labor |
| $\chi$ | Lump-sum transfer |
| $g$ | Rate of technical progress |
| $\beta$ | Subjective discount factor |
| $\tilde{\beta}$ | Growth-adjusted discount factor: $\beta(1+g)^{\eta(1-\mu)-1}$ |
| $\eta$ | Consumption weight in Cobb-Douglas utility |
| $\mu$ | Coefficient of relative risk aversion on the composite good |
| $\pi_{e,e'}$ | Markov transition probability from state $e$ to $e'$ |
| $V(\tilde{a}, e)$ | Monolithic value function |
| $V^{\texttt{LS}}(\tilde{a}, e)$ | Value function at the `labor-supply` stage |
| $W^{\texttt{cons}}(m, l)$ | Value function at the `consumption` stage |
| $v^{\texttt{disc}}(\tilde{a}')$ | Value function at the `disc` stage |
| $\bar{V}(\tilde{a}'; e)$ | Expected continuation value from the connector |


## 2. Why Multiple Stages?

### Why the monolithic formulation is problematic

In the Bellman equation from Section 1, the household jointly chooses consumption $\tilde{c}$ and leisure $l$ in a single $\max$ operator. Treating this as one undifferentiated optimization is costly for several reasons:

1. **Two-dimensional search.** The monolithic formulation requires maximizing over $(\tilde{c}, l)$ simultaneously. On a grid, this means evaluating the objective at every $(c_i, l_j)$ pair — a 2D grid search whose cost grows as $N_c \times N_l$. A sequential decomposition replaces this with two 1D problems.

2. **Coupled first-order conditions.** Because the Cobb-Douglas utility $u(\tilde{c}, l) = (\tilde{c}^{\eta} l^{1-\eta})^{1-\mu}/(1-\mu)$ is non-separable, the FOC for consumption involves $l$ and the FOC for leisure involves $\tilde{c}$. In the monolithic formulation these are solved as a simultaneous system. A stage decomposition makes the dependence explicit and sequential.

3. **Hidden structure.** The monolithic $\max$ hides the fact that the labor/leisure decision has a distinct economic role: it determines how much total resources the household has available. Once labor income is determined, the remaining consumption-saving choice is structurally identical to the single-control problem solved in `SolvingMicroDSOPs` Sections 1–11.

### Timing within a period

There is a natural ordering of events within each period:

1. **Arrival:** The household enters period $t$ with assets $\tilde{a}_t$. The productivity shock $e_t$ is realized (drawn from the Markov chain).
2. **Labor-supply decision:** Observing $(\tilde{a}_t, e_t)$, the household chooses leisure $l_t$ (equivalently, labor hours $h_t = 1 - l_t$). This pins down labor income $\bar{w} e_t (1 - l_t)$ and total resources:

$$m_t \;=\; (1+\bar{r})\,\tilde{a}_t \;+\; \bar{w}\,e_t\,(1-l_t) \;+\; \chi$$

3. **Consumption-saving decision:** Given resources $m_t$ and the already-chosen $l_t$, the household chooses consumption $\tilde{c}_t$. Savings are residual: $\tilde{a}_{t+1} = (m_t - \tilde{c}_t)/(1+g)$.
4. **Exit:** The household carries $\tilde{a}_{t+1}$ into period $t+1$.

The labor choice constrains the consumption choice by setting the budget, but not the reverse. This asymmetry makes labor-first, consumption-second the natural ordering.

### How decomposition helps (DDSL perspective)

This structure maps directly onto the stage/perch architecture from `SolvingMicroDSOPs` Section 13:

- **Modularity.** The consumption-saving stage is the *same* `cons-noshocks` module used in the single-control problem. Only the preceding labor-supply stage is new. Adding labor supply to the model does not require rewriting the consumption solver.

- **Analogy to the portfolio problem.** In Sections 12–13 of `SolvingMicroDSOPs`, the portfolio share $\varsigma$ is a second control that affects the budget set (through portfolio returns) but does not directly enter the utility function. In our model, leisure $l$ plays a similar structural role — it determines total resources — but additionally enters the utility function. The decomposition strategy is the same: solve the "resource-determining" control first, then pass the resulting budget to the consumption stage.

- **Computational gains.** Each stage is a 1D optimization. The labor-supply stage solves a single FOC for $l$ given $(\tilde{a}, e)$ and the continuation value from the consumption stage. The consumption stage can use the Endogenous Grid Method (EGM), exactly as in the single-control case, conditional on $l$.


## 3. Stage Decomposition

We decompose each period into three stages plus an inter-period connector. The forward (chronological) ordering within a period is:

$$(\tilde{a}, e) \xrightarrow{\texttt{labor-supply}} (m, l) \xrightarrow{\texttt{consumption}} \tilde{a}' \xrightarrow{\texttt{disc}} \tilde{a}' \xrightarrow{\texttt{connector}} (\tilde{a}', e')_{\text{next period}}$$

The backward pass (used for solving) proceeds **right-to-left**: `disc` is solved first from next period's value, then `consumption`, then `labor-supply`.

---

### Stage 1: `labor-supply`

This stage handles the arrival of the productivity shock and the labor/leisure choice.

| Perch | Indicator | State | Value function(s) | Explanation |
|-------|-----------|-------|-------------------|-------------|
| Arrival | $\prec$ | $(\tilde{a}, e)$ | $V^{\texttt{LS}}_{\prec}(\tilde{a}, e) = V^{\texttt{LS}}_{\sim}(\tilde{a}, e)$ | The Markov shock $e$ is **realized** between the arrival perch and the decision perch. |
| Decision(s) | $\sim$ | $(\tilde{a}, e)$ | $V^{\texttt{LS}}_{\sim}(\tilde{a}, e)$ | Choose leisure $l \in [0,1]$; this determines resources $m = (1+\bar{r})\tilde{a} + \bar{w} e (1-l) + \chi$. |
| Continuation | $\succ$ | $(m, l)$ | $V^{\texttt{LS}}_{\succ}(m, l) = W^{\texttt{cons}}_{\prec}(m, l)$ | Exit the stage carrying total resources and the chosen leisure into the `consumption` stage. |

**Immediate reward:** None. The full period utility $u(\tilde{c}, l)$ cannot be evaluated here because consumption $\tilde{c}$ is not yet chosen. The reward is deferred entirely to the `consumption` stage.

**Value function:**

$$V^{\texttt{LS}}(\tilde{a}, e) \;=\; \max_{l \in [0,1]} \; W^{\texttt{cons}}\!\Big(\underbrace{(1+\bar{r})\tilde{a} + \bar{w}\,e\,(1-l) + \chi}_{m(l)},\; l\Big)$$

where $W^{\texttt{cons}}$ is the value function from the `consumption` stage below.

**FOC for leisure (intratemporal condition at this stage).**

The first-order condition for $l$ requires differentiating $W^{\texttt{cons}}(m(l), l)$ with respect to $l$:

$$\frac{dW^{\texttt{cons}}}{dl} \;=\; \frac{\partial W^{\texttt{cons}}}{\partial m}\cdot\frac{dm}{dl} \;+\; \frac{\partial W^{\texttt{cons}}}{\partial l} \;=\; 0$$

Since $m(l) = (1+\bar{r})\tilde{a} + \bar{w}\,e\,(1-l) + \chi$, we have $dm/dl = -\bar{w}\,e$. By the envelope theorem, the partial derivatives of $W^{\texttt{cons}}$ at the optimal $\tilde{c}^*$ are:

$$\frac{\partial W^{\texttt{cons}}}{\partial m} = u_{\tilde{c}}(\tilde{c}^*, l) = \eta\,\tilde{c}^{*\,\eta(1-\mu)-1}\,l^{(1-\eta)(1-\mu)}$$

$$\frac{\partial W^{\texttt{cons}}}{\partial l} = u_l(\tilde{c}^*, l) = (1-\eta)\,\tilde{c}^{*\,\eta(1-\mu)}\,l^{(1-\eta)(1-\mu)-1}$$

Substituting into the FOC and rearranging:

$$(1-\eta)\,\tilde{c}^{*\,\eta(1-\mu)}\,l^{(1-\eta)(1-\mu)-1} \;=\; \bar{w}\,e\;\cdot\;\eta\,\tilde{c}^{*\,\eta(1-\mu)-1}\,l^{(1-\eta)(1-\mu)}$$

Dividing both sides by $\tilde{c}^{*\,\eta(1-\mu)-1}\,l^{(1-\eta)(1-\mu)-1}$ gives the standard Cobb-Douglas intratemporal condition:

$$\boxed{\;\frac{\tilde{c}^*}{l} \;=\; \frac{\eta}{1-\eta}\;\bar{w}\,e\;}$$

This result is clean and economically intuitive: the optimal consumption-to-leisure ratio is proportional to the effective wage $\bar{w}\,e$. A household facing a higher productivity state $e$ tilts toward more consumption and less leisure — working more to exploit the higher wage.

This intratemporal condition, together with the consumption Euler equation from the `consumption` stage, fully characterizes the solution. The stage decomposition reveals that each condition lives in its own stage: the Euler equation is solved inside `consumption` (by EGM), and the intratemporal condition is solved inside `labor-supply` (by 1D root-finding or optimization), with the two stages communicating through $W^{\texttt{cons}}(m, l)$.

**Role:** This stage is analogous to the `portable` stage in `SolvingMicroDSOPs`, which also precedes consumption and determines the effective budget. The difference is that in the portfolio problem, $\varsigma$ affects the budget through stochastic returns but does not enter the utility function. Here, $l$ affects the budget through labor income **and** enters the utility function through the Cobb-Douglas aggregator.

---

### Stage 2: `consumption`

This stage handles the consumption-saving decision, taking the labor/leisure choice as given.

| Perch | Indicator | State | Value function(s) | Explanation |
|-------|-----------|-------|-------------------|-------------|
| Arrival | $\prec$ | $(m, l)$ | $W^{\texttt{cons}}_{\prec}(m, l) = W^{\texttt{cons}}_{\sim}(m, l)$ | Enter stage with total resources and the leisure choice inherited from `labor-supply`; no new shocks arrive here. |
| Decision(s) | $\sim$ | $(m, l)$ | $W^{\texttt{cons}}_{\sim}(m, l)$ | Choose consumption $\tilde{c} \in [0,m]$, taking leisure $l$ as fixed. |
| Continuation | $\succ$ | $\tilde{a}' = (m-\tilde{c})/(1+g)$ | $W^{\texttt{cons}}_{\succ}(\tilde{a}'; l) = v^{\texttt{disc}}_{\prec}(\tilde{a}')$ | Exit the stage with end-of-period assets, which are passed into the `disc` stage. |

**Immediate reward:** The full period utility is realized here:

$$u(\tilde{c}, l) \;=\; \frac{\left(\tilde{c}^{\;\eta}\, l^{\;1-\eta}\right)^{1-\mu}}{1-\mu}$$

Because utility is non-separable in $(\tilde{c}, l)$, the leisure parameter $l$ enters this stage as a **fixed input** from the preceding stage, not as a choice variable.

**Value function:**

$$W^{\texttt{cons}}(m, l) \;=\; \max_{\tilde{c}} \left\{ \frac{(\tilde{c}^{\;\eta}\, l^{\;1-\eta})^{1-\mu}}{1-\mu} \;+\; v^{\texttt{disc}}\!\left(\frac{m - \tilde{c}}{1+g}\right) \right\}$$

where $v^{\texttt{disc}}$ is the value from the `disc` stage below.

**FOC for consumption (Euler equation at this stage):**

Differentiating with respect to $\tilde{c}$ and setting equal to the marginal continuation value:

$$u_{\tilde{c}}(\tilde{c}, l) \;=\; \frac{1}{1+g}\; {v^{\texttt{disc}}}'(\tilde{a}')$$

$$\eta\, \tilde{c}^{\,\eta(1-\mu)-1}\, l^{\,(1-\eta)(1-\mu)} \;=\; \frac{1}{1+g}\; {v^{\texttt{disc}}}'(\tilde{a}')$$

**EGM inversion (step-by-step).** For a given $l$, this is a one-dimensional equation in $\tilde{c}$. The EGM strategy starts from a grid of $\tilde{a}'$ values:

1. For each $\tilde{a}'$ on the grid, evaluate the right-hand side ${v^{\texttt{disc}}}'(\tilde{a}')/(1+g)$.
2. Invert the marginal utility to find the unique $\tilde{c}$ satisfying the FOC:

$$\tilde{c}^*(\tilde{a}'; l) \;=\; \left(\frac{{v^{\texttt{disc}}}'(\tilde{a}')}{\eta\,(1+g)\; l^{\,(1-\eta)(1-\mu)}}\right)^{\frac{1}{\eta(1-\mu) - 1}}$$

3. Recover total resources from the budget constraint:

$$m \;=\; \tilde{c}^* \;+\; (1+g)\,\tilde{a}'$$

This gives a set of $(m, \tilde{c}^*)$ pairs — the endogenous gridpoints. Interpolating on these pairs produces the consumption policy function $\tilde{c}^*(m; l)$ for each value of $l$.

**Note on non-separability:** The factor $l^{(1-\eta)(1-\mu)}$ acts as a multiplicative scaling constant on marginal utility. For a given $l$, the consumption stage behaves like a standard CRRA consumption-savings problem with an effective coefficient of relative risk aversion $1 - \eta(1-\mu)$ on consumption alone. The key point is that $l$ enters only as a parameter, not as a choice variable. This is why the decomposition works despite the non-separable utility: the consumption stage's EGM machinery is essentially unchanged, just rescaled by a leisure-dependent constant.

---

### Stage 3: `disc`

This stage applies time discounting. It has no control and no shocks.

| Perch | Indicator | State | Value function(s) | Explanation |
|-------|-----------|-------|-------------------|-------------|
| Arrival | $\prec$ | $\tilde{a}'$ | $v^{\texttt{disc}}_{\prec}(\tilde{a}')$ | Enter stage with end-of-period assets coming from the `consumption` stage. |
| Decision(s) | $\sim$ | $\tilde{a}'$ | $v^{\texttt{disc}}_{\sim}(\tilde{a}') = v^{\texttt{disc}}_{\prec}(\tilde{a}')$ | No choice is made in this stage; it is purely a value transformation. |
| Continuation | $\succ$ | $\tilde{a}'$ | $v^{\texttt{disc}}_{\succ}(\tilde{a}') = \tilde{\beta}\,\bar V(\tilde{a}'; e)$ | Pass assets unchanged to the connector after applying discounting to the continuation value. |

**Value function (pure rescaling):**

$$v^{\texttt{disc}}(\tilde{a}') \;=\; \tilde{\beta} \;\cdot\; \bar{V}(\tilde{a}')$$

where $\bar{V}(\tilde{a}')$ is the expected next-period value delivered by the connector, and $\tilde{\beta} = \beta(1+g)^{\eta(1-\mu)-1}$ is the growth-adjusted discount factor.

This is identical in structure to the `disc` stage in `SolvingMicroDSOPs`, just with $\tilde{\beta}$ replacing $\beta\Gamma^{1-\rho}$.

---

### Connector (period $t$ → period $t+1$)

The connector is **not** a stage. It links the continuation perch of the `disc` stage in period $t$ to the arrival perch of `labor-supply` in period $t+1$.

**What it does:**

1. Takes end-of-period assets $\tilde{a}'$ and the current Markov state $e$ from period $t$.
2. Computes the expected value over next period's productivity realization $e'$:

$$\bar{V}(\tilde{a}';\, e) \;=\; \sum_{e'} \pi_{e,e'}\; V^{\texttt{LS}}_{t+1}(\tilde{a}',\, e')$$

The connector is where the Markov transition probabilities $\pi_{e,e'}$ appear. The mechanical state transition for assets is trivial here (assets pass through unchanged; the return $(1+\bar{r})$ on $\tilde{a}'$ will be applied at the arrival of next period's `labor-supply` stage when resources $m$ are computed).

---

### Composition: full period

Composing stages **right-to-left** (backward pass order):

$$\tilde{a}' \;\xrightarrow{\texttt{connector}}\; \bar{V}(\tilde{a}'; e) \;\xrightarrow{\texttt{disc}}\; v^{\texttt{disc}}(\tilde{a}') = \tilde{\beta}\,\bar{V}(\tilde{a}'; e) \;\xrightarrow{\texttt{consumption}}\; W^{\texttt{cons}}(m, l) \;\xrightarrow{\texttt{labor-supply}}\; V^{\texttt{LS}}(\tilde{a}, e)$$

In the backward solver, each period $t$ is solved as:

1. **Connector:** Given $V^{\texttt{LS}}_{t+1}(\cdot,\cdot)$ from the already-solved period $t+1$, form the expected continuation $\bar{V}(\tilde{a}'; e) = \sum_{e'}\pi_{e,e'}\,V^{\texttt{LS}}_{t+1}(\tilde{a}', e')$.
2. **`disc`:** Multiply by $\tilde{\beta}$: $\;v^{\texttt{disc}}(\tilde{a}'; e) = \tilde{\beta}\,\bar{V}(\tilde{a}'; e)$.
3. **`consumption`:** For each $l$, solve for optimal $\tilde{c}^*(m; l)$ via EGM using the consumption FOC. This produces $W^{\texttt{cons}}(m, l)$.
4. **`labor-supply`:** For each $(\tilde{a}, e)$, solve a 1D optimization over $l$ to find $l^*(\tilde{a}, e)$ that maximizes $W^{\texttt{cons}}(m(l), l)$. This yields $V^{\texttt{LS}}_t(\tilde{a}, e)$.

**Comparison with `SolvingMicroDSOPs` Section 13:**

| `SolvingMicroDSOPs` (`port-cons`) | `OptimumDebt` |
|-----------------------------------|---------------|
| `portable` (returns + shocks + $\varsigma$ optimization) | `labor-supply` (shock + $l$ optimization) |
| `cons-noshocks` (EGM for $c$) | `consumption` (EGM for $\tilde{c}$, conditional on $l$) |
| `disc` ($\beta\Gamma^{1-\rho}$) | `disc` ($\tilde{\beta}$) |
| Connector: between-period | Connector: Markov expectation $\sum_{e'}\pi_{e,e'}$ |

---

### Key structural insight

In the monolithic Bellman equation (Section 1), leisure $l$ plays a **dual role** that is hidden inside a single $\max$ operator:

1. **Resource channel:** $l$ determines labor income $\bar{w}\,e\,(1-l)$, and thus the total resources available for consumption and saving.
2. **Utility channel:** $l$ enters the Cobb-Douglas utility $(\tilde{c}^{\eta} l^{1-\eta})^{1-\mu}/(1-\mu)$, directly contributing to the household's welfare.

The stage decomposition makes this dual role structurally explicit. The `labor-supply` stage resolves the tension between these two channels: working more (lower $l$) produces more income but reduces utility from leisure. Its FOC — the intratemporal condition $\tilde{c}/l = \frac{\eta}{1-\eta}\bar{w}\,e$ — is the precise point where the two channels balance. Meanwhile, the `consumption` stage handles the purely intertemporal trade-off (consume today vs. save for tomorrow), taking the resource level set by the labor stage as given.

This parallels the Imai-Keane (2004) decomposition, where labor supply affects **both** current resources **and** the law of motion for human capital — a coupling that the monolithic formulation hides inside a single $\max$ operator but which is structurally explicit in the two-stage decomposition. In our model, the coupling is between labor income and leisure utility rather than between labor income and human capital accumulation, but the architectural insight is the same: decomposing into stages reveals the distinct economic channels through which a single control variable operates.


## 4. Comparison with the Monolithic Formulation

### Equivalence

The staged decomposition is a change in **representation**, not in economics. To verify, substitute the stage value functions from Section 3 back into one another:

$$V^{\texttt{LS}}(\tilde{a}, e) = \max_l \; W^{\texttt{cons}}(m(l), l) = \max_l \max_{\tilde{c}} \left\{ u(\tilde{c}, l) + \tilde{\beta} \sum_{e'}\pi_{e,e'} V^{\texttt{LS}}(\tilde{a}', e') \right\}$$

Because the constraints on $\tilde{c}$ and $l$ are independent (choosing $l$ determines $m$, and $\tilde{c} \in [0, m]$ is then a separate feasible set), the nested $\max_l \max_{\tilde{c}}$ is equivalent to the joint $\max_{\tilde{c}, l}$ in the monolithic Bellman from Section 1. The two formulations yield the same policy functions and the same value function.

### What the decomposition makes clearer

1. **Distinct roles of the two controls.** In the monolithic formulation, $\tilde{c}$ and $l$ appear symmetrically inside a single $\max$, obscuring the fact that they do different things. The stage view makes explicit that $l$ determines the budget (resources $m$) while $\tilde{c}$ allocates that budget between current consumption and saving.

2. **Which FOC belongs where.** The monolithic problem has two coupled first-order conditions. The decomposition assigns each to a specific stage:

   | FOC | Stage | Equation |
   |-----|-------|----------|
   | **Consumption Euler equation** | `consumption` | $\eta\,\tilde{c}^{\eta(1-\mu)-1} l^{(1-\eta)(1-\mu)} = \frac{1}{1+g}{v^{\texttt{disc}}}'(\tilde{a}')$ |
   | **Intratemporal labor-leisure** | `labor-supply` | $\frac{\partial W^{\texttt{cons}}}{\partial m}\cdot \bar{w}\,e = \frac{\partial W^{\texttt{cons}}}{\partial l}$ |

   The consumption Euler equation is internal to the `consumption` stage and can be solved by EGM. The intratemporal condition is the optimality condition at the `labor-supply` stage: the marginal value of the extra income from working one more unit ($\frac{\partial W}{\partial m} \cdot \bar{w}e$) must equal the marginal cost in foregone leisure ($\frac{\partial W}{\partial l}$).

3. **How non-separability propagates.** The Cobb-Douglas term $l^{(1-\eta)(1-\mu)}$ appears in the consumption FOC as a multiplicative constant. The decomposition makes this coupling visible: the `consumption` stage inherits $l$ from the `labor-supply` stage and uses it to scale marginal utility. In the monolithic formulation, this dependence is buried inside the simultaneous system.

### Numerical solution strategy suggested by the decomposition

The stage view naturally suggests an **iterative two-step algorithm** within each backward-induction period:

1. **Inner loop (consumption stage):** For a grid of $(m, l)$ pairs, solve the 1D consumption problem by EGM. This is fast because it inverts the marginal utility analytically.
2. **Outer loop (labor-supply stage):** For each $(\tilde{a}, e)$, search over $l$ to maximize $W^{\texttt{cons}}(m(l), l)$. This is a 1D root-finding or optimization problem, using the consumption policy from step 1.

This is more efficient than the monolithic alternative (2D grid search over $(\tilde{c}, l)$ pairs, or solving two coupled FOCs simultaneously by Newton's method). It also mirrors the solution strategy used by Aiyagari and McGrattan themselves: they solve the intratemporal condition for optimal $l^*(x, e)$ given the consumption policy, then use that to reduce the problem to a single Euler equation (their residual function $R(x, i; \alpha)$ in Section III).

### Alternative decompositions

One could reverse the ordering: `consumption` first, then `labor-supply`. In this variant the agent would first choose $\tilde{c}$ from initial wealth, then choose $l$ to determine how much labor income to add. This is less natural because:

- The consumption decision would be made before the agent knows total resources (since labor income hasn't been determined yet).
- The `labor-supply` stage would need to handle both the leisure component of utility and the resource augmentation, losing the clean separation of "budget determination" and "budget allocation."

The `labor-supply → consumption` ordering chosen here follows the natural economic logic and matches the `portable → cons-noshocks` pattern in `SolvingMicroDSOPs`.

### Limitations

The main complication relative to the `SolvingMicroDSOPs` portfolio problem is **non-separable utility**. In the portfolio problem, $\varsigma$ does not enter the utility function, so the consumption stage is a pure `cons-noshocks` module with no modification. Here, $l$ enters the Cobb-Douglas utility, so the consumption stage must receive $l$ as an additional input. This means the consumption stage is not *identical* to the off-the-shelf `cons-noshocks` — it is a parametrically modified version. The modification is mild (a multiplicative scaling of marginal utility), but it does reduce the plug-and-play reusability that the architecture aims for. At the same time, the Cobb-Douglas structure keeps the coupling tractable: the intratemporal condition equates the ratio of marginal utilities to the effective wage, so consumption and leisure satisfy a simple proportionality condition rather than an arbitrary nonlinear system.